In [1]:


# CELL 1 — INSTALL + CURRENT SESSION TIMER

from pathlib import Path
import time

MARKER = Path("/kaggle/working/L8_session_start.txt")

# Start a fresh time budget for this Kaggle session.
MARKER.write_text(str(time.time()))

print("Session start:", time.ctime())
print("Maximum session: 11h30")
print("Training safe stop: 11h15")

%pip -q install \
    "lerobot[smolvla,dataset]==0.6.1" \
    "transformers==5.5.0" \
    "av==15.1.0" \
    "pyarrow" \
    "huggingface_hub" \
    "accelerate" \
    "pandas" \
    "psutil"


Session start: Tue Nov  3 12:21:07 2026
Maximum session: 11h30
Training safe stop: 11h15
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nilearn 0.14.1 requires packaging>=26.0.0, but you have packaging 25.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:

import os
import re
import torch
from kaggle_secrets import UserSecretsClient

assert torch.cuda.is_available(), "Enable GPU T4 x2"
assert torch.cuda.device_count() == 2, (
    f"Expected 2 GPUs, got {torch.cuda.device_count()}"
)

raw = os.getenv("HF_TOKEN") or UserSecretsClient().get_secret("HF_TOKEN")

matches = re.findall(r"hf_[A-Za-z0-9]{20,}", str(raw))
assert len(matches) == 1, "Invalid HF_TOKEN in Kaggle Secrets"

os.environ["HF_TOKEN"] = matches[0]

for i in range(2):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

print("Hugging Face token: OK")


GPU 0: Tesla T4
GPU 1: Tesla T4
Hugging Face token: OK


In [3]:

%%writefile /kaggle/working/lfm2_l8_train_ddp.py
import os
import gc
import json
import time
import math
import random
import hashlib
import contextlib
import datetime
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist

from torch.nn.parallel import DistributedDataParallel as DDP
from PIL import Image
from huggingface_hub import HfApi, hf_hub_download
from transformers import (
    AutoProcessor,
    Lfm2VlForConditionalGeneration,
    DynamicCache,
)
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

# ============================================================
# CELL 3 — VERIFIED LFM2 L8 EXPERIMENT CONFIGURATION
# ============================================================

MODEL_NAME = "L8"

DATASET_ID = "MinhDuk/Chunking_296_Skynet_combined"
BACKBONE_ID = "LiquidAI/LFM2-VL-450M"
EXPERT_ID = "lerobot/smolvla_base"

KEEP_LAYERS = 8
SEED = 42

CHUNK = 50
ACTION_DIM = 6
MAX_ACTION_DIM = 32

STEPS = 4000
VAL_EVERY = 500
LOG_EVERY = 20

# 2 GPUs x 1 sample/GPU x 8 accumulation
GLOBAL_BATCH = 16
GRAD_ACCUM = 8

FLOW_STEPS = 10
VIS_TOKENS_PER_CAM = 48
TEST_NOISE_DRAWS = 3

LR = 1e-4
WARMUP = 200
GRAD_CLIP = 1.0

SESSION_LIMIT_MIN = 690
STOP_BUFFER_MIN = 15
VAL_RESERVE_MIN = 25

SAVE_EVERY_UPDATES = 25
SAVE_EVERY_SECONDS = 600

CAM_KEYS = (
    "observation.images.cam_head",
    "observation.images.cam_side",
    "observation.images.cam_wrist",
)

# New experiment: NEVER resume an old, incompatible L8.
OUT = Path(
    "/kaggle/working/L8_FAIR_V2_296_70_10_20"
)
CSV_DIR = OUT / "csv"
CKPT_DIR = OUT / "checkpoints"
AUDIT_DIR = OUT / "audit"

LAST_FILE = CKPT_DIR / "L8_last.pt"
BEST_FILE = CKPT_DIR / "L8_best.pt"

for folder in (CSV_DIR, CKPT_DIR, AUDIT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

assert MODEL_NAME == "L8"
assert BACKBONE_ID == "LiquidAI/LFM2-VL-450M"
assert KEEP_LAYERS == 8
assert GLOBAL_BATCH == 2 * GRAD_ACCUM
assert STEPS == 4000
assert FLOW_STEPS == 10

print("L8 CONFIGURATION: PASS", flush=True)


Writing /kaggle/working/lfm2_l8_train_ddp.py


In [4]:

%%writefile -a /kaggle/working/lfm2_l8_train_ddp.py

# ============================================================
# DISTRIBUTED INITIALIZATION
# ============================================================

dist.init_process_group(
    "nccl",
    timeout=datetime.timedelta(hours=3),
)

RANK = dist.get_rank()
WORLD_SIZE = dist.get_world_size()
LOCAL_RANK = int(os.environ["LOCAL_RANK"])

assert WORLD_SIZE == 2

torch.cuda.set_device(LOCAL_RANK)
DEVICE = torch.device("cuda", LOCAL_RANK)

AMP_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

USE_SCALER = AMP_DTYPE == torch.float16

random.seed(SEED + RANK)
np.random.seed(SEED + RANK)
torch.manual_seed(SEED + RANK)
torch.cuda.manual_seed(SEED + RANK)

HF_TOKEN = os.environ["HF_TOKEN"]
HF_TOKEN = re.findall(r"hf_[A-Za-z0-9]{20,}", HF_TOKEN)[0]

api = HfApi(token=HF_TOKEN)

DATA_REVISION = api.repo_info(
    DATASET_ID, repo_type="dataset"
).sha

BACKBONE_REVISION = api.model_info(BACKBONE_ID).sha
EXPERT_REVISION = api.model_info(EXPERT_ID).sha

def log(*args):
    if RANK == 0:
        print(*args, flush=True)

def write_csv(rows, path):
    if RANK == 0 and rows:
        temp = Path(str(path) + ".tmp")
        pd.DataFrame(rows).to_csv(temp, index=False)
        os.replace(temp, path)

def to_cpu_tree(obj):
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: to_cpu_tree(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return type(obj)([to_cpu_tree(v) for v in obj])
    return obj

log("LFM2 L8 | DDP ranks:", WORLD_SIZE)
log("Backbone:", BACKBONE_ID)
log("Layers:", KEEP_LAYERS)
log("AMP:", AMP_DTYPE)


Appending to /kaggle/working/lfm2_l8_train_ddp.py


In [5]:

%%writefile -a /kaggle/working/lfm2_l8_train_ddp.py

# ============================================================
# DATASET + FIXED SPLIT
# ============================================================

meta_file = hf_hub_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    filename="meta/info.json",
    revision=DATA_REVISION,
    token=HF_TOKEN,
)

INFO = json.loads(Path(meta_file).read_text(encoding="utf-8"))
FPS = float(INFO["fps"])

assert int(INFO["total_episodes"]) == 296
assert int(INFO["total_frames"]) == 434945

repo_files = api.list_repo_files(
    DATASET_ID,
    repo_type="dataset",
    revision=DATA_REVISION,
)

episode_files = sorted(
    f for f in repo_files
    if f.startswith("meta/episodes/") and f.endswith(".parquet")
)

if episode_files:
    all_ids = []
    for name in episode_files:
        p = hf_hub_download(
            repo_id=DATASET_ID,
            repo_type="dataset",
            filename=name,
            revision=DATA_REVISION,
            token=HF_TOKEN,
        )
        all_ids.extend(
            pq.read_table(
                p, columns=["episode_index"]
            ).column("episode_index").to_pylist()
        )
    EPISODE_IDS = sorted(set(map(int, all_ids)))
else:
    p = hf_hub_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        filename="meta/episodes.jsonl",
        revision=DATA_REVISION,
        token=HF_TOKEN,
    )
    EPISODE_IDS = sorted(
        int(x)
        for x in pd.read_json(
            p, lines=True
        )["episode_index"].unique()
    )

assert len(EPISODE_IDS) == 296

shuffled = np.random.default_rng(SEED).permutation(
    EPISODE_IDS
).tolist()

TRAIN_EPS = sorted(shuffled[:207])
VAL_EPS = sorted(shuffled[207:237])
TEST_EPS = sorted(shuffled[237:])

assert (len(TRAIN_EPS), len(VAL_EPS), len(TEST_EPS)) == (
    207, 30, 59
)

SPLIT = {
    "dataset": DATASET_ID,
    "revision": DATA_REVISION,
    "seed": SEED,
    "train": TRAIN_EPS,
    "validation": VAL_EPS,
    "test": TEST_EPS,
}

SPLIT_HASH = hashlib.sha256(
    json.dumps(SPLIT, sort_keys=True).encode()
).hexdigest()

if RANK == 0:
    (AUDIT_DIR / "dataset_split.json").write_text(
        json.dumps(
            {**SPLIT, "sha256": SPLIT_HASH},
            indent=2,
        )
    )

    write_csv(
        [
            {"episode_index": e, "Split": split}
            for split, eps in (
                ("train", TRAIN_EPS),
                ("validation", VAL_EPS),
                ("test", TEST_EPS),
            )
            for e in eps
        ],
        CSV_DIR / "01_dataset_split.csv",
    )

delta = {
    "action": [
        round(i / FPS, 6)
        for i in range(CHUNK)
    ]
}

DS = LeRobotDataset(
    DATASET_ID,
    episodes=EPISODE_IDS,
    delta_timestamps=delta,
    revision=DATA_REVISION,
    token=HF_TOKEN,
    download_videos=True,
    video_backend="pyav",
)

frame_df = (
    DS.hf_dataset.select_columns(
        ["episode_index", "frame_index"]
    )
    .to_pandas()
    .reset_index(drop=True)
)

assert len(frame_df) == len(DS)

WINDOWS = {}

for ep, group in frame_df.groupby("episode_index"):
    group = group.sort_values("frame_index")
    last_frame = int(group.frame_index.max())

    eligible = group[
        group.frame_index <= last_frame - CHUNK + 1
    ]

    WINDOWS[int(ep)] = eligible.index.to_numpy(dtype=np.int64)

assert all(len(WINDOWS[e]) >= 3 for e in EPISODE_IDS)

def fixed_windows(episodes):
    plan = []
    for ep in episodes:
        indices = WINDOWS[ep]
        positions = (
            np.linspace(0, len(indices) - 1, 5)[1:-1]
            .round()
            .astype(int)
        )
        for idx in indices[positions]:
            row = frame_df.iloc[int(idx)]
            plan.append({
                "episode_index": int(ep),
                "frame_index": int(row.frame_index),
                "index": int(idx),
            })
    return plan

VAL_PLAN = fixed_windows(VAL_EPS)
TEST_PLAN = fixed_windows(TEST_EPS)

assert len(VAL_PLAN) == 90
assert len(TEST_PLAN) == 177

if RANK == 0:
    write_csv(VAL_PLAN, AUDIT_DIR / "validation_plan.csv")
    write_csv(TEST_PLAN, AUDIT_DIR / "test_plan.csv")

log("Dataset:", len(DS), "frames")
log("Split:", len(TRAIN_EPS), len(VAL_EPS), len(TEST_EPS))
log("Split SHA256:", SPLIT_HASH)


Appending to /kaggle/working/lfm2_l8_train_ddp.py


In [6]:

%%writefile -a /kaggle/working/lfm2_l8_train_ddp.py

# ============================================================
# CELL 6 — TRAIN-ONLY NORMALIZATION AND SAMPLE INPUT
# ============================================================

if RANK == 0:
    data_files = sorted(
        f for f in repo_files
        if f.startswith("data/")
        and f.endswith(".parquet")
    )

    assert data_files

    all_actions = []
    all_states = []

    for filename in data_files:
        path = hf_hub_download(
            repo_id=DATASET_ID,
            repo_type="dataset",
            filename=filename,
            revision=DATA_REVISION,
            token=HF_TOKEN,
        )

        df = pq.read_table(
            path,
            columns=[
                "episode_index",
                "action",
                "observation.state",
            ],
        ).to_pandas()

        df = df[
            df["episode_index"].isin(TRAIN_EPS)
        ]

        if len(df):
            all_actions.extend(df["action"].tolist())
            all_states.extend(
                df["observation.state"].tolist()
            )

    A = np.stack(all_actions).astype(np.float32)
    S = np.stack(all_states).astype(np.float32)

    assert A.shape[1] == ACTION_DIM
    assert S.shape[1] == ACTION_DIM
    assert np.isfinite(A).all()
    assert np.isfinite(S).all()

    np.savez(
        AUDIT_DIR / "train_only_normalization.npz",
        action_mean=A.mean(axis=0),
        action_std=A.std(axis=0).clip(min=1e-5),
        state_mean=S.mean(axis=0),
        state_std=S.std(axis=0).clip(min=1e-5),
    )

    log("Normalization frames:", len(A))

    del A, S, all_actions, all_states
    gc.collect()

dist.barrier()

with np.load(
    AUDIT_DIR / "train_only_normalization.npz",
    allow_pickle=False,
) as stats:
    AM = torch.tensor(
        stats["action_mean"], device=DEVICE
    )
    AS = torch.tensor(
        stats["action_std"], device=DEVICE
    )
    SM = torch.tensor(
        stats["state_mean"], device=DEVICE
    )
    SS = torch.tensor(
        stats["state_std"], device=DEVICE
    )

assert all(
    torch.isfinite(x).all()
    for x in (AM, AS, SM, SS)
)

PROTOCOL = {
    "model": MODEL_NAME,
    "seed": SEED,
    "dataset": DATASET_ID,
    "dataset_revision": DATA_REVISION,
    "split_sha256": SPLIT_HASH,
    "backbone": BACKBONE_ID,
    "backbone_revision": BACKBONE_REVISION,
    "backbone_layers": KEEP_LAYERS,
    "backbone_frozen": True,
    "expert": EXPERT_ID,
    "expert_revision": EXPERT_REVISION,
    "expert_pretrained": True,
    "expert_trainable": True,
    "adaptive_conditioner_trainable": True,
    "conditioner_design": "F12_AdaptiveConditionalGatedKV",
    "visual_tokens_per_camera": VIS_TOKENS_PER_CAM,
    "chunk": CHUNK,
    "action_dim": ACTION_DIM,
    "max_action_dim": MAX_ACTION_DIM,
    "flow_steps": FLOW_STEPS,
    "optimizer_updates": STEPS,
    "effective_batch": GLOBAL_BATCH,
    "parallelism": "DDP",
    "gpu_count": WORLD_SIZE,
    "grad_accum_per_gpu": GRAD_ACCUM,
    "validation_interval": VAL_EVERY,
    "learning_rate": LR,
    "warmup": WARMUP,
    "gradient_clip": GRAD_CLIP,
    "seed": SEED,
    "loss": "FM_MSE_6D_mean",
    "best_checkpoint_metric": "Val_MAE",
    "test_noise_draws": TEST_NOISE_DRAWS,
}

if RANK == 0:
    (AUDIT_DIR / "protocol_config.json").write_text(
        json.dumps(PROTOCOL, indent=2),
        encoding="utf-8",
    )


def to_pil(raw):
    # Same conversion as the FAST F12 notebook.
    x = torch.as_tensor(
        raw
    ).detach().cpu().float()

    if x.ndim == 4:
        assert x.shape[0] == 1
        x = x[0]

    assert x.ndim == 3, x.shape

    if x.shape[-1] in (3, 4) and x.shape[0] != 3:
        x = x.permute(2, 0, 1)

    x = x[:3]

    if float(x.max()) > 1.5:
        x = x / 255.0

    array = (
        x.clamp(0, 1)
        .permute(1, 2, 0)
        .contiguous()
        .numpy()
    )

    return Image.fromarray(
        (array * 255).round().astype(np.uint8)
    )


def prepare_sample(index):
    raw = DS[int(index)]

    assert "task" in raw, "Missing dataset task"

    task = str(raw["task"]).strip()
    assert task

    if "action_is_pad" in raw:
        assert not bool(
            torch.as_tensor(
                raw["action_is_pad"]
            ).any()
        )

    state = torch.as_tensor(
        raw["observation.state"],
        dtype=torch.float32,
        device=DEVICE,
    ).reshape(1, ACTION_DIM)

    actions = torch.as_tensor(
        raw["action"],
        dtype=torch.float32,
        device=DEVICE,
    ).reshape(1, CHUNK, ACTION_DIM)

    assert torch.isfinite(state).all()
    assert torch.isfinite(actions).all()

    return {
        "images": [
            raw[key] for key in CAM_KEYS
        ],
        "task": task,
        "state": (state - SM) / SS,
        "actions": (actions - AM) / AS,
        "raw_actions": actions,
    }

log("CELL 6: PASS")


Appending to /kaggle/working/lfm2_l8_train_ddp.py


In [7]:

%%writefile -a /kaggle/working/lfm2_l8_train_ddp.py

# ============================================================
# CELL 7 — LFM2-VL 450M / PRETRAINED / FROZEN / 8 LAYERS
# ============================================================

log("Loading LiquidAI/LFM2-VL-450M...")

backbone = Lfm2VlForConditionalGeneration.from_pretrained(
    BACKBONE_ID,
    revision=BACKBONE_REVISION,
    token=HF_TOKEN,
    torch_dtype=AMP_DTYPE,
    low_cpu_mem_usage=True,
)

processor = AutoProcessor.from_pretrained(
    BACKBONE_ID,
    revision=BACKBONE_REVISION,
    token=HF_TOKEN,
)

language = backbone.model.language_model

assert hasattr(language, "layers")
assert len(language.layers) == 16, (
    f"Expected 16 layers, found {len(language.layers)}"
)

# Preserve the FIRST eight pretrained language layers.
original_layer_types = list(
    backbone.config.text_config.layer_types
)

assert len(original_layer_types) == 16

language.layers = nn.ModuleList(
    list(language.layers[:KEEP_LAYERS])
)

language.config.num_hidden_layers = KEEP_LAYERS
language.config.layer_types = (
    original_layer_types[:KEEP_LAYERS]
)

backbone.config.text_config.num_hidden_layers = (
    KEEP_LAYERS
)
backbone.config.text_config.layer_types = (
    original_layer_types[:KEEP_LAYERS]
)

# The language-modeling output head is not used for
# representation extraction.
backbone.lm_head = nn.Identity()

backbone.to(DEVICE)
backbone.requires_grad_(False)
backbone.eval()

LFM_DIM = int(
    backbone.config.text_config.hidden_size
)

assert LFM_DIM == 1024
assert len(backbone.model.language_model.layers) == 8
assert not any(
    p.requires_grad
    for p in backbone.parameters()
)

log("LFM2 language layers:", KEEP_LAYERS)
log("Original layer types:", original_layer_types)
log("Kept layer types:", original_layer_types[:8])
log("Hidden size:", LFM_DIM)
log("Frozen:", True)


@torch.no_grad()
def lfm_features(sample):
    """
    Frozen LFM2-VL feature extractor.

    Input:
        3 images, one task instruction.

    Output:
        [1, seq_len, 1024]

    Each camera contributes exactly 48 pooled
    visual tokens, matching the FAST F12 token budget.
    """

    images = [
        to_pil(image)
        for image in sample["images"]
    ]

    assert len(images) == 3

    tokenizer = processor.tokenizer

    # Text and image features are interleaved in
    # head / side / wrist order.
    text_parts = [
        "Head camera: ",
        "\nSide camera: ",
        "\nWrist camera: ",
        "\n" + sample["task"],
    ]

    embeddings = []

    for camera_idx, image in enumerate(images):
        # Text label before its corresponding camera.
        ids = tokenizer(
            text_parts[camera_idx],
            add_special_tokens=False,
            return_tensors="pt",
        )["input_ids"].to(DEVICE)

        if ids.shape[1] > 0:
            embeddings.append(
                language.embed_tokens(ids)
            )

        image_inputs = processor.image_processor(
            images=image,
            return_tensors="pt",
        )

        required = (
            "pixel_values",
            "spatial_shapes",
            "pixel_attention_mask",
        )

        assert all(
            k in image_inputs
            for k in required
        ), (
            f"LFM image preprocessing mismatch: "
            f"{list(image_inputs.keys())}"
        )

        prepared = {}

        for key in required:
            tensor = image_inputs[key].to(DEVICE)

            if key == "pixel_values":
                tensor = tensor.to(AMP_DTYPE)

            prepared[key] = tensor

        with torch.autocast(
            "cuda",
            dtype=AMP_DTYPE,
        ):
            image_output = backbone.get_image_features(
                **prepared,
                return_dict=True,
            )

        # LFM2-VL's multimodal projector produces
        # image features in language hidden dimension.
        visual = image_output.pooler_output

        if isinstance(visual, (list, tuple)):
            assert len(visual) > 0
            visual = torch.cat(
                visual, dim=0
            )

        if visual.ndim == 3:
            visual = visual.reshape(
                -1, visual.shape[-1]
            )

        assert visual.ndim == 2
        assert visual.shape[-1] == LFM_DIM
        assert visual.shape[0] > 0

        visual = F.adaptive_avg_pool1d(
            visual.transpose(0, 1).unsqueeze(0),
            VIS_TOKENS_PER_CAM,
        ).transpose(1, 2)

        assert visual.shape == (
            1,
            VIS_TOKENS_PER_CAM,
            LFM_DIM,
        )

        embeddings.append(visual)

    # Append the actual dataset instruction.
    ids = tokenizer(
        text_parts[3],
        add_special_tokens=False,
        return_tensors="pt",
    )["input_ids"].to(DEVICE)

    assert ids.shape[1] > 0

    embeddings.append(
        language.embed_tokens(ids)
    )

    sequence = torch.cat(
        embeddings, dim=1
    )

    attention_mask = torch.ones(
        sequence.shape[:2],
        dtype=torch.long,
        device=DEVICE,
    )

    with torch.autocast(
        "cuda",
        dtype=AMP_DTYPE,
    ):
        result = language(
            inputs_embeds=sequence,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )

    hidden = result.last_hidden_state

    assert hidden.ndim == 3
    assert hidden.shape[0] == 1
    assert hidden.shape[-1] == LFM_DIM
    assert torch.isfinite(hidden).all()

    return hidden.detach()


# Verify the actual representation pathway before
# initialization of the Action Expert.
probe = prepare_sample(
    int(WINDOWS[TRAIN_EPS[0]][0])
)

features = lfm_features(probe)

assert features.shape[-1] == 1024

log("3-camera feature shape:", tuple(features.shape))
log("LFM2 8/16 layers: PASS")
log("LFM2 image projector: PASS")
log("LFM2 frozen feature extraction: PASS")

del probe, features
gc.collect()


Appending to /kaggle/working/lfm2_l8_train_ddp.py


In [8]:

%%writefile -a /kaggle/working/lfm2_l8_train_ddp.py

# ============================================================
# CELL 8 — PRETRAINED ACTION EXPERT + F12 CONDITIONER
# ============================================================

policy = SmolVLAPolicy.from_pretrained(
    EXPERT_ID,
    revision=EXPERT_REVISION,
).to(DEVICE)

m = policy.model
ve = m.vlm_with_expert
expert = ve.lm_expert

assert ve.attention_mode == "cross_attn"
assert int(ve.self_attn_every_n_layers) == 2
assert m.config.chunk_size == CHUNK
assert m.config.max_action_dim == MAX_ACTION_DIM
assert m.config.max_state_dim == MAX_ACTION_DIM

N_EXPERT_LAYERS = len(expert.layers)

# Freeze all pretrained policy parameters first.
policy.requires_grad_(False)

TRAIN_MODULES = {
    "expert": expert,
    "state_proj": m.state_proj,
    "action_in_proj": m.action_in_proj,
    "action_out_proj": m.action_out_proj,
    "action_time_mlp_in": m.action_time_mlp_in,
    "action_time_mlp_out": m.action_time_mlp_out,
}

for module in TRAIN_MODULES.values():
    module.float()
    module.requires_grad_(True)

ve.vlm.requires_grad_(False)
ve.vlm.eval()

m.config.use_cache = True

# Use original SmolVLA VLM KV geometry.
vlm_config = ve.vlm.config.text_config

KV_HEADS = int(
    vlm_config.num_key_value_heads
)
HEAD_DIM = int(
    vlm_config.head_dim
)
KV_WIDTH = KV_HEADS * HEAD_DIM

assert KV_HEADS > 0
assert HEAD_DIM > 0

# ============================================================
# SAME ADAPTIVE CONDITIONAL GATED KV DESIGN AS FAST F12
# Only input_dim differs because LFM2 hidden_size = 1024.
# ============================================================

class AdaptiveConditionalGatedKV(nn.Module):
    def __init__(
        self,
        input_dim,
        state_dim,
        kv_dim,
        kv_heads,
        head_dim,
        num_layers,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.kv_dim = kv_dim
        self.kv_heads = kv_heads
        self.head_dim = head_dim
        self.num_layers = num_layers

        self.mapper = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.SiLU(),
            nn.LayerNorm(input_dim),
        )

        self.state_bridge = nn.Linear(
            state_dim, input_dim
        )

        self.kv_projection = nn.Linear(
            input_dim, 2 * kv_dim
        )

        # ONE gate per layer, shared across K and V.
        self.gate_network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.SiLU(),
            nn.Linear(128, num_layers),
        )

        self.layer_bias = nn.Parameter(
            torch.zeros(
                num_layers, 2, kv_dim
            )
        )

    def forward(self, vl_features, state_features):
        x = self.mapper(
            vl_features.float()
        )

        state_token = self.state_bridge(
            state_features.float()
        ).unsqueeze(1)

        x = torch.cat(
            [x, state_token],
            dim=1,
        )

        B, P, _ = x.shape

        gates = torch.sigmoid(
            self.gate_network(
                x.mean(dim=1)
            )
        )

        base_k, base_v = (
            self.kv_projection(x).chunk(
                2, dim=-1
            )
        )

        cache = DynamicCache()

        for layer_id in range(self.num_layers):
            gate = gates[
                :, layer_id
            ].reshape(B, 1, 1)

            k = (
                base_k
                + self.layer_bias[layer_id, 0]
            ) * gate

            v = (
                base_v
                + self.layer_bias[layer_id, 1]
            ) * gate

            k = k.reshape(
                B,
                P,
                self.kv_heads,
                self.head_dim,
            ).transpose(1, 2).contiguous()

            v = v.reshape(
                B,
                P,
                self.kv_heads,
                self.head_dim,
            ).transpose(1, 2).contiguous()

            # Exactly like FAST F12:
            # no custom RoPE transformation here.
            cache.update(
                k, v, layer_id
            )

        return cache


conditioner = AdaptiveConditionalGatedKV(
    input_dim=LFM_DIM,
    state_dim=m.state_proj.out_features,
    kv_dim=KV_WIDTH,
    kv_heads=KV_HEADS,
    head_dim=HEAD_DIM,
    num_layers=N_EXPERT_LAYERS,
).to(DEVICE).float()

TRAIN_MODULES["conditioner"] = conditioner

# ============================================================
# EXPERT FORWARD
# ============================================================

class L8Trainable(nn.Module):
    def __init__(self, model, conditioner):
        super().__init__()
        self.m = model
        self.conditioner = conditioner

    def make_cache(self, features, state):
        padded_state = F.pad(
            state.float(),
            (0, MAX_ACTION_DIM - ACTION_DIM),
        )

        state_features = self.m.state_proj(
            padded_state
        )

        return self.conditioner(
            features,
            state_features,
        )

    def velocity(
        self, features, state, x_t, timestep
    ):
        cache = self.make_cache(
            features, state
        )

        prefix_len = cache.layers[0].keys.shape[2]

        prefix_mask = torch.ones(
            x_t.shape[0],
            prefix_len,
            device=x_t.device,
            dtype=torch.bool,
        )

        velocity = self.m.denoise_step(
            prefix_pad_masks=prefix_mask,
            past_key_values=cache,
            x_t=x_t,
            timestep=timestep,
        )

        assert velocity.shape == x_t.shape
        return velocity

    def forward(
        self,
        features,
        state,
        actions,
        noise,
        timestep,
    ):
        x0 = F.pad(
            actions.float(),
            (0, MAX_ACTION_DIM - ACTION_DIM),
        )

        timestep = timestep.float().reshape(
            actions.shape[0]
        )

        t = timestep[:, None, None]

        x_t = (1 - t) * x0 + t * noise
        target_velocity = noise - x0

        predicted = self.velocity(
            features,
            state,
            x_t,
            timestep,
        )

        # Same 6D flow matching objective as F12.
        return (
            predicted[:, :, :ACTION_DIM].float()
            - target_velocity[:, :, :ACTION_DIM].float()
        ).square().mean()


system = L8Trainable(
    m, conditioner
).to(DEVICE)

params = [
    p for p in system.parameters()
    if p.requires_grad
]

assert params
assert len(set(map(id, params))) == len(params)

assert all(
    p.dtype == torch.float32
    for p in params
)

assert not any(
    p.requires_grad
    for p in backbone.parameters()
)

total_trainable = sum(
    p.numel() for p in params
)

expert_params = sum(
    p.numel()
    for p in expert.parameters()
)

assert expert_params > 0
assert all(
    p.requires_grad
    for p in expert.parameters()
)

# ============================================================
# CSV 02 — MATCH FAST F12 ARCHITECTURE SCHEMA
# ============================================================

if RANK == 0:
    used_total = (
        sum(
            p.numel()
            for p in backbone.parameters()
        )
        + sum(
            p.numel()
            for module in TRAIN_MODULES.values()
            for p in module.parameters()
        )
    )

    pd.DataFrame([{
        "Model": MODEL_NAME,
        "Backbone": BACKBONE_ID,
        "Layers": KEEP_LAYERS,
        "Expert_Layers": N_EXPERT_LAYERS,
        "Total_Used_Params_M": used_total / 1e6,
        "Trainable_Params_M": total_trainable / 1e6,
        "Expert_Params_M": expert_params / 1e6,
        "Conditioner_Params_M": sum(
            p.numel()
            for p in conditioner.parameters()
        ) / 1e6,
        "Backbone_Frozen": True,
        "Expert_Trainable": True,
        "Dataset_Revision": DATA_REVISION,
        "Split_SHA256": SPLIT_HASH,
    }]).to_csv(
        CSV_DIR / "02_model_architecture.csv",
        index=False,
    )

    pd.DataFrame([
        {
            "Model": MODEL_NAME,
            "Module": name,
            "Total_Params": sum(
                p.numel()
                for p in module.parameters()
            ),
            "Trainable_Params": sum(
                p.numel()
                for p in module.parameters()
                if p.requires_grad
            ),
            "Pretrained": name != "conditioner",
        }
        for name, module in TRAIN_MODULES.items()
    ]).to_csv(
        AUDIT_DIR / "trainable_modules.csv",
        index=False,
    )

# ============================================================
# 2-GPU DISTRIBUTED TRAINING
# ============================================================

ddp = DDP(
    system,
    device_ids=[LOCAL_RANK],
    output_device=LOCAL_RANK,
    find_unused_parameters=False,
    broadcast_buffers=False,
)

optimizer = torch.optim.AdamW(
    params,
    lr=LR,
    weight_decay=0.01,
)

def lr_factor(step):
    if step < WARMUP:
        return (step + 1) / WARMUP

    progress = (
        (step - WARMUP)
        / max(1, STEPS - WARMUP)
    )

    return 0.5 * (
        1 + math.cos(math.pi * progress)
    )

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_factor,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_SCALER,
)

# Rank 0 will generate the shared global sample
# schedule for both GPUs.
rng = np.random.default_rng(
    SEED if RANK == 0 else SEED + 1
)

log("LFM2 frozen: PASS")
log("SmolVLA expert pretrained: PASS")
log("F12 conditioner architecture: PASS")
log("DDP initialized: PASS")
log("Trainable parameters:", total_trainable)


Appending to /kaggle/working/lfm2_l8_train_ddp.py


In [9]:

%%writefile -a /kaggle/working/lfm2_l8_train_ddp.py

# ============================================================
# CHECKPOINT SYSTEM
# ============================================================

train_log = []
val_log = []
sel_log = []

start_step = 0
best_mae = float("inf")
prev_elapsed = 0.0

def model_states():
    return {
        name: to_cpu_tree(module.state_dict())
        for name, module in TRAIN_MODULES.items()
    }

def load_model_states(states):
    assert set(states) == set(TRAIN_MODULES)

    for name, module in TRAIN_MODULES.items():
        module.load_state_dict(
            states[name],
            strict=True,
        )

def collect_rank_rng():
    local = {
        "python": random.getstate(),
        "numpy": rng.bit_generator.state,
        "torch": torch.get_rng_state().cpu(),
        "cuda": torch.cuda.get_rng_state(DEVICE).cpu(),
    }

    gathered = [None] * WORLD_SIZE
    dist.all_gather_object(gathered, local)
    return gathered

def restore_rng(state):
    random.setstate(state["python"])
    rng.bit_generator.state = state["numpy"]

    torch.set_rng_state(
        state["torch"].cpu()
    )

    torch.cuda.set_rng_state(
        state["cuda"].cpu(),
        device=DEVICE,
    )

def save_ckpt(
    path, step, best, elapsed, include_optimizer=True
):
    # Collective: both GPU ranks must call this function.
    rank_rng = collect_rank_rng()

    if RANK == 0:
        payload = {
            "model": MODEL_NAME,
            "step": int(step),
            "best_mae": float(best),
            "train_wall_s": float(elapsed),
            "dataset_revision": DATA_REVISION,
            "backbone_revision": BACKBONE_REVISION,
            "expert_revision": EXPERT_REVISION,
            "split_sha256": SPLIT_HASH,
            "states": model_states(),
            "rank_rng": rank_rng,
            "world_size": WORLD_SIZE,
        }

        if include_optimizer:
            payload.update({
                "optimizer": to_cpu_tree(
                    optimizer.state_dict()
                ),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
            })

        temporary = Path(str(path) + ".tmp")

        torch.save(payload, temporary)
        os.replace(temporary, path)

        log(
            "[SAVED]",
            path.name,
            "step", step,
            "best MAE", best,
        )

    dist.barrier()

# ============================================================
# RESUME EXISTING L8 CHECKPOINT
# ============================================================

if LAST_FILE.exists():
    ckpt = torch.load(
        LAST_FILE,
        map_location="cpu",
        weights_only=False,
    )

    assert ckpt["model"] == "L8"
    assert ckpt["dataset_revision"] == DATA_REVISION
    assert ckpt["backbone_revision"] == BACKBONE_REVISION
    assert ckpt["expert_revision"] == EXPERT_REVISION
    assert ckpt["split_sha256"] == SPLIT_HASH
    assert ckpt["world_size"] == WORLD_SIZE

    load_model_states(ckpt["states"])

    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])

    restore_rng(ckpt["rank_rng"][RANK])

    start_step = int(ckpt["step"])
    best_mae = float(ckpt["best_mae"])
    prev_elapsed = float(ckpt["train_wall_s"])

    if RANK == 0:
        for path, target in (
            (
                CSV_DIR / "04_training_history.csv",
                train_log,
            ),
            (
                CSV_DIR / "05_validation_history.csv",
                val_log,
            ),
            (
                CSV_DIR / "06_checkpoint_selection.csv",
                sel_log,
            ),
        ):
            if path.exists():
                df = pd.read_csv(path)

                target.extend(
                    df[df["Step"] <= start_step]
                    .to_dict("records")
                )

    log("[RESUMED] L8 from step", start_step)

    del ckpt
    gc.collect()

else:
    log("[NEW L8] Starting fresh training")

assert 0 <= start_step <= STEPS



Appending to /kaggle/working/lfm2_l8_train_ddp.py


In [10]:

%%writefile -a /kaggle/working/lfm2_l8_train_ddp.py

# ============================================================
# CELL 10 — DETERMINISTIC VALIDATION
# SAME METRICS AND WINDOW PROTOCOL AS FAST F12
# ============================================================

def deterministic_seed(item, draw=0):
    return int(
        SEED * 1000003
        + int(item["episode_index"]) * 10007
        + int(item["frame_index"]) * 31
        + int(draw)
    )


def noise_tensor(seed):
    generator = torch.Generator(
        device=DEVICE
    ).manual_seed(int(seed))

    return torch.randn(
        (1, CHUNK, MAX_ACTION_DIM),
        generator=generator,
        device=DEVICE,
    )


@torch.no_grad()
def predict_action(features, state, seed):
    cache = system.make_cache(
        features, state
    )

    prefix_len = cache.layers[0].keys.shape[2]

    prefix_mask = torch.ones(
        1,
        prefix_len,
        device=DEVICE,
        dtype=torch.bool,
    )

    x_t = noise_tensor(seed)

    for i in range(FLOW_STEPS):
        timestep = torch.full(
            (1,),
            1.0 - i / FLOW_STEPS,
            dtype=torch.float32,
            device=DEVICE,
        )

        with torch.autocast(
            "cuda", dtype=AMP_DTYPE
        ):
            velocity = m.denoise_step(
                prefix_pad_masks=prefix_mask,
                past_key_values=cache,
                x_t=x_t,
                timestep=timestep,
            )

        x_t = (
            x_t - velocity.float() / FLOW_STEPS
        )

    return (
        x_t[:, :, :ACTION_DIM].float() * AS + AM
    )


@torch.no_grad()
def fixed_flow_loss(sample, seed):
    features = lfm_features(sample)

    noise = noise_tensor(seed)

    timestep = torch.full(
        (1,),
        0.5,
        dtype=torch.float32,
        device=DEVICE,
    )

    with torch.autocast(
        "cuda", dtype=AMP_DTYPE
    ):
        loss = system(
            features,
            sample["state"],
            sample["actions"],
            noise,
            timestep,
        )

    return float(loss.item())


@torch.no_grad()
def validation(step):
    system.eval()
    backbone.eval()
    ve.vlm.eval()

    rows = []
    t0 = time.perf_counter()

    for item in VAL_PLAN:
        sample = prepare_sample(
            item["index"]
        )

        seed = deterministic_seed(item)

        features = lfm_features(sample)

        predicted = predict_action(
            features,
            sample["state"],
            seed,
        )

        ground_truth = sample["raw_actions"]

        difference = (
            predicted.float()
            - ground_truth.float()
        )

        mae = float(
            difference.abs().mean().item()
        )

        mse = float(
            difference.square().mean().item()
        )

        fm_loss = fixed_flow_loss(
            sample, seed
        )

        rows.append({
            "MAE": mae,
            "MSE": mse,
            "Flow_Loss": fm_loss,
        })

        del sample, features, predicted

    df = pd.DataFrame(rows)

    assert len(df) == len(VAL_PLAN) == 90
    assert np.isfinite(
        df[["MAE", "MSE", "Flow_Loss"]]
        .to_numpy()
    ).all()

    torch.cuda.synchronize()

    result = {
        "Model": MODEL_NAME,
        "Step": int(step),
        "Val_MAE": float(df["MAE"].mean()),
        "Val_MSE": float(df["MSE"].mean()),
        "Val_RMSE": math.sqrt(
            float(df["MSE"].mean())
        ),
        "Val_Flow_Loss": float(
            df["Flow_Loss"].mean()
        ),
        "Val_Windows": len(VAL_PLAN),
        "Val_Episodes": len(VAL_EPS),
        "Validation_s": (
            time.perf_counter() - t0
        ),
    }

    system.train()
    backbone.eval()
    ve.vlm.eval()

    return result


Appending to /kaggle/working/lfm2_l8_train_ddp.py


In [11]:

%%writefile -a /kaggle/working/lfm2_l8_train_ddp.py

# ============================================================
# CELL 11 — TWO-GPU GRADIENT SMOKE TEST
# ZERO OPTIMIZER UPDATES
# ============================================================

if start_step == 0:
    system.train()
    backbone.eval()
    ve.vlm.eval()

    probe_index = int(
        WINDOWS[TRAIN_EPS[RANK]][0]
    )

    sample = prepare_sample(probe_index)
    features = lfm_features(sample)

    assert features.ndim == 3
    assert features.shape[-1] == LFM_DIM

    optimizer.zero_grad(set_to_none=True)

    noise = torch.randn(
        (1, CHUNK, MAX_ACTION_DIM),
        device=DEVICE,
    )

    timestep = torch.full(
        (1,), 0.5,
        device=DEVICE,
        dtype=torch.float32,
    )

    with torch.autocast(
        "cuda", dtype=AMP_DTYPE
    ):
        loss = ddp(
            features,
            sample["state"],
            sample["actions"],
            noise,
            timestep,
        )

    assert torch.isfinite(loss)

    scaler.scale(loss).backward()

    audit_rows = []

    for name, module in TRAIN_MODULES.items():
        gradients = [
            p.grad
            for p in module.parameters()
            if p.requires_grad and p.grad is not None
        ]

        assert gradients, (
            f"Missing gradient: {name}"
        )

        assert all(
            torch.isfinite(g).all()
            for g in gradients
        ), f"Nonfinite gradient: {name}"

        norm = sum(
            float(g.float().norm().item())
            for g in gradients
        )

        assert norm > 0, (
            f"Zero gradient: {name}"
        )

        audit_rows.append({
            "Model": MODEL_NAME,
            "Rank": RANK,
            "GPU": LOCAL_RANK,
            "Module": name,
            "Tensors_With_Grad": len(gradients),
            "Grad_Norm_Sum": norm,
        })

    gathered = [
        None for _ in range(WORLD_SIZE)
    ]

    dist.all_gather_object(
        gathered, audit_rows
    )

    if RANK == 0:
        rows = [
            row
            for part in gathered
            for row in part
        ]

        pd.DataFrame(rows).to_csv(
            AUDIT_DIR / "gradient_audit.csv",
            index=False,
        )

    optimizer.zero_grad(set_to_none=True)

    dist.barrier()

    log("TWO-GPU SMOKE TEST: PASS")
    log("Gradient audit: PASS")
    log("Optimizer updates: 0")

    del sample, features, loss, noise
    gc.collect()

else:
    log(
        "[RESUME] Smoke test skipped "
        "to preserve checkpoint RNG."
    )


Appending to /kaggle/working/lfm2_l8_train_ddp.py


In [12]:

%%writefile -a /kaggle/working/lfm2_l8_train_ddp.py

# ============================================================
# CELL 12 — LFM2 L8 FAIR TWO-GPU TRAINING
# 4000 UPDATES, BATCH 16, 11h30 SESSION
# ============================================================

marker = Path(
    "/kaggle/working/L8_session_start.txt"
)

absolute_start = float(
    marker.read_text().strip()
)

stop_by = (
    absolute_start
    + (SESSION_LIMIT_MIN - STOP_BUFFER_MIN) * 60
)

completed = start_step
last_saved = start_step
last_save_time = time.monotonic()

train_start = time.perf_counter()


def total_training_seconds():
    return (
        prev_elapsed
        + time.perf_counter() - train_start
    )


def flush_logs():
    if RANK == 0:
        write_csv(
            train_log,
            CSV_DIR / "04_training_history.csv",
        )

        write_csv(
            val_log,
            CSV_DIR / "05_validation_history.csv",
        )

        write_csv(
            sel_log,
            CSV_DIR / "06_checkpoint_selection.csv",
        )


def save_latest(step, reason):
    global last_saved, last_save_time

    flush_logs()

    save_ckpt(
        LAST_FILE,
        step,
        best_mae,
        total_training_seconds(),
        include_optimizer=True,
    )

    last_saved = step
    last_save_time = time.monotonic()

    log(
        f"[SAVED] step={step} reason={reason}"
    )


def evaluate_checkpoint(step):
    global best_mae

    if RANK == 0:
        result = validation(step)

        val_log.append(result)

        improved = (
            result["Val_MAE"] < best_mae
        )

        if improved:
            best_mae = result["Val_MAE"]

        sel_log.append({
            "Model": MODEL_NAME,
            "Step": step,
            "Val_MAE": result["Val_MAE"],
            "Val_Flow_Loss": result["Val_Flow_Loss"],
            "Is_Best_At_This_Step": bool(improved),
            "Best_Val_MAE_So_Far": best_mae,
        })

        log(
            f"[VAL {step}] "
            f"MAE={result['Val_MAE']:.6f} "
            f"RMSE={result['Val_RMSE']:.6f} "
            f"FM={result['Val_Flow_Loss']:.6f} "
            f"BEST={best_mae:.6f}"
        )

        selection = [
            best_mae, improved
        ]

    else:
        selection = [None, None]

    # Both ranks participate in synchronization.
    dist.broadcast_object_list(
        selection, src=0
    )

    best_mae = float(selection[0])
    improved = bool(selection[1])

    if improved:
        save_ckpt(
            BEST_FILE,
            step,
            best_mae,
            total_training_seconds(),
            include_optimizer=False,
        )

    save_latest(
        step, "validation_completed"
    )


def get_global_sample_indices():
    """
    Rank 0 draws all 16 indices in the SAME order
    as the single-GPU FAST F12 episode sampler.

    8 samples -> GPU 0
    8 samples -> GPU 1
    """

    indices = torch.empty(
        GLOBAL_BATCH,
        dtype=torch.long,
        device=DEVICE,
    )

    if RANK == 0:
        selected = []

        for _ in range(GLOBAL_BATCH):
            episode = int(
                rng.choice(TRAIN_EPS)
            )

            index = int(
                rng.choice(WINDOWS[episode])
            )

            selected.append(index)

        indices.copy_(
            torch.tensor(
                selected,
                dtype=torch.long,
                device=DEVICE,
            )
        )

    dist.broadcast(indices, src=0)

    return indices[
        RANK * GRAD_ACCUM:
        (RANK + 1) * GRAD_ACCUM
    ]


log("=" * 65)
log("LFM2 L8 — FAIR TWO GPU TRAINING")
log("Backbone:", BACKBONE_ID)
log("Layers:", KEEP_LAYERS)
log("GPUs:", WORLD_SIZE)
log("Effective batch:", GLOBAL_BATCH)
log("Gradient accumulation/GPU:", GRAD_ACCUM)
log("Start step:", start_step)
log("Target updates:", STEPS)
log("Session limit:", SESSION_LIMIT_MIN, "minutes")
log("Auto-stop buffer:", STOP_BUFFER_MIN, "minutes")
log("Checkpoint interval:", SAVE_EVERY_UPDATES)
log("=" * 65)

# Catch-up validation after a previous time-safe stop.
if (
    start_step > 0
    and start_step % VAL_EVERY == 0
):
    if RANK == 0:
        already_validated = any(
            int(row["Step"]) == start_step
            for row in val_log
        )
    else:
        already_validated = False

    status = [already_validated]
    dist.broadcast_object_list(status, src=0)

    if (
        not status[0]
        and time.time()
        + VAL_RESERVE_MIN * 60 < stop_by
    ):
        evaluate_checkpoint(start_step)

try:
    for step in range(
        start_step + 1,
        STEPS + 1,
    ):

        # Synchronized time-safe stop.
        stop_flag = torch.tensor(
            [int(time.time() >= stop_by)],
            dtype=torch.int32,
            device=DEVICE,
        )

        dist.all_reduce(
            stop_flag,
            op=dist.ReduceOp.MAX,
        )

        if stop_flag.item():
            log("[TIME SAFE STOP]")
            break

        system.train()
        backbone.eval()
        ve.vlm.eval()

        optimizer.zero_grad(set_to_none=True)

        started_step = time.perf_counter()
        micro_losses = []

        # Exactly 16 global samples per update.
        local_indices = (
            get_global_sample_indices()
        )

        for micro in range(GRAD_ACCUM):
            index = int(
                local_indices[micro].item()
            )

            sample = prepare_sample(index)

            with torch.no_grad():
                features = lfm_features(sample)

            # Sync only on the final microbatch.
            sync_context = (
                ddp.no_sync()
                if micro < GRAD_ACCUM - 1
                else contextlib.nullcontext()
            )

            with sync_context:
                with torch.autocast(
                    "cuda",
                    dtype=AMP_DTYPE,
                ):
                    noise = torch.randn(
                        (
                            1,
                            CHUNK,
                            MAX_ACTION_DIM,
                        ),
                        device=DEVICE,
                    )

                    timestep = m.sample_time(
                        1, DEVICE
                    )

                    loss = ddp(
                        features,
                        sample["state"],
                        sample["actions"],
                        noise,
                        timestep,
                    )

                assert torch.isfinite(loss), (
                    f"Nonfinite loss at step {step}"
                )

                scaler.scale(
                    loss / GRAD_ACCUM
                ).backward()

            micro_losses.append(
                float(loss.detach().item())
            )

            del sample, features
            del noise, timestep, loss

        scaler.unscale_(optimizer)

        grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                params,
                max_norm=GRAD_CLIP,
                error_if_nonfinite=True,
            )
        )

        old_scale = scaler.get_scale()

        scaler.step(optimizer)
        scaler.update()

        if (
            USE_SCALER
            and scaler.get_scale() < old_scale
        ):
            raise FloatingPointError(
                f"Skipped optimizer update {step}"
            )

        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        torch.cuda.synchronize()

        step_seconds = (
            time.perf_counter() - started_step
        )

        mean_loss = torch.tensor(
            [
                float(
                    np.mean(micro_losses)
                )
            ],
            device=DEVICE,
            dtype=torch.float32,
        )

        dist.all_reduce(
            mean_loss,
            op=dist.ReduceOp.AVG,
        )

        completed = step

        if RANK == 0:
            peak_vram = (
                torch.cuda.max_memory_allocated()
                / 1024**3
            )

            train_log.append({
                "Model": MODEL_NAME,
                "Step": step,
                "Samples_Seen": (
                    step * GLOBAL_BATCH
                ),
                "Train_Flow_Loss": float(
                    mean_loss.item()
                ),
                "Learning_Rate": float(
                    optimizer.param_groups[0]["lr"]
                ),
                "Grad_Norm": float(
                    grad_norm.item()
                ),
                "Step_s": step_seconds,
                "Peak_VRAM_GiB": peak_vram,
                "GPU_Count": WORLD_SIZE,
            })

        if (
            step == start_step + 1
            or step % LOG_EVERY == 0
        ):
            flush_logs()

            log(
                f"[L8 {step:04d}/{STEPS}] "
                f"loss={mean_loss.item():.5f} "
                f"step_s={step_seconds:.2f} "
                f"elapsed_min="
                f"{(time.time()-absolute_start)/60:.1f}"
            )

        due_save = (
            step - last_saved
            >= SAVE_EVERY_UPDATES
            or time.monotonic() - last_save_time
            >= SAVE_EVERY_SECONDS
            or step % VAL_EVERY == 0
            or step == STEPS
            or time.time() >= stop_by
        )

        if due_save:
            save_latest(
                step, "periodic_or_time"
            )

        if time.time() >= stop_by:
            log(
                "[STOP] Checkpoint saved. "
                "Ready for next Kaggle session."
            )
            break

        if step % VAL_EVERY == 0:
            remaining_seconds = (
                stop_by - time.time()
            )

            if (
                remaining_seconds
                < VAL_RESERVE_MIN * 60
            ):
                log(
                    "[STOP] Validation deferred "
                    "to next session."
                )
                break

            evaluate_checkpoint(step)

    if completed > last_saved:
        save_latest(
            completed, "session_end"
        )

    flush_logs()

    # ========================================================
    # CSV 03 — MATCH FAST F12 SCHEMA
    # ========================================================

    if RANK == 0:
        elapsed = total_training_seconds()

        trainable_count = sum(
            p.numel() for p in params
        )

        pd.DataFrame([{
            "Model": MODEL_NAME,
            "Training_Scope": "Independent L8",
            "Optimizer_Updates": completed,
            "Samples_Seen": (
                completed * GLOBAL_BATCH
            ),
            "Effective_Batch": GLOBAL_BATCH,
            "Train_Wall_s": elapsed,
            "Train_Wall_Hours": elapsed / 3600,
            "Peak_VRAM_GiB": (
                torch.cuda.max_memory_allocated()
                / 1024**3
            ),
            "GPU": torch.cuda.get_device_name(
                DEVICE
            ),
            "Updates_per_s": (
                completed / max(elapsed, 1e-9)
            ),
            "Trainable_Params_M": (
                trainable_count / 1e6
            ),
            "GPU_Count": WORLD_SIZE,
            "Parallelism": "DDP",
            "Remaining_Updates": (
                STEPS - completed
            ),
        }]).to_csv(
            CSV_DIR / "03_training_efficiency.csv",
            index=False,
        )

        log("=" * 65)
        log("LFM2 L8 TRAINING SESSION FINISHED")
        log("Completed updates:", completed)
        log("Remaining:", STEPS - completed)
        log("Best validation MAE:", best_mae)
        log("Checkpoint:", LAST_FILE)
        log("=" * 65)

    dist.barrier()

finally:
    if dist.is_initialized():
        dist.destroy_process_group()


Appending to /kaggle/working/lfm2_l8_train_ddp.py


In [13]:

# ============================================================
# CELL 12.5 — PATCH FINAL TEST INTO L8 DDP SCRIPT
# MATCH FAST F12 TEST SCHEMA: CSV 07, 08, 09
# ============================================================

from pathlib import Path

script_path = Path(
    "/kaggle/working/lfm2_l8_train_ddp.py"
)

source = script_path.read_text(encoding="utf-8")

marker = "    dist.barrier()\n\nfinally:"

assert source.count(marker) == 1, (
    "Cannot find final training boundary. "
    "Check that Cell 12 ran successfully."
)

assert "L8_FINAL_TEST_PATCH_V1" not in source, (
    "Final test patch already installed."
)

patch = r'''
    # ========================================================
    # L8_FINAL_TEST_PATCH_V1
    # HELD-OUT TEST: 59 EPISODES / 177 WINDOWS / 531 PREDICTIONS
    # ========================================================

    if RANK == 0 and completed == STEPS:

        if time.time() + 20 * 60 >= stop_by:
            log(
                "[TEST DEFERRED] Not enough session time. "
                "Resume in another Kaggle session."
            )

        else:
            assert BEST_FILE.exists(), (
                "Missing L8_best.pt"
            )

            best_payload = torch.load(
                BEST_FILE,
                map_location="cpu",
                weights_only=False,
            )

            assert best_payload["model"] == "L8"
            assert (
                best_payload["split_sha256"] == SPLIT_HASH
            )

            load_model_states(
                best_payload["states"]
            )

            BEST_STEP = int(
                best_payload["step"]
            )

            system.eval()
            backbone.eval()
            ve.vlm.eval()

            TEST_NOISE_DRAWS = 3

            test_rows = []
            test_flow_losses = []

            log(
                "[FINAL TEST] Best step:",
                BEST_STEP,
                "Test windows:",
                len(TEST_PLAN),
            )

            for window_id, item in enumerate(TEST_PLAN):

                sample = prepare_sample(
                    item["index"]
                )

                features = lfm_features(sample)

                base_seed = deterministic_seed(item)

                with torch.no_grad():

                    noise = noise_tensor(base_seed)

                    timestep = torch.full(
                        (1,),
                        0.5,
                        device=DEVICE,
                        dtype=torch.float32,
                    )

                    with torch.autocast(
                        "cuda",
                        dtype=AMP_DTYPE,
                    ):
                        fm_loss = system(
                            features,
                            sample["state"],
                            sample["actions"],
                            noise,
                            timestep,
                        )

                    test_flow_losses.append(
                        float(fm_loss.item())
                    )

                    for draw in range(TEST_NOISE_DRAWS):

                        seed = base_seed + draw

                        torch.cuda.synchronize()
                        tic = time.perf_counter()

                        pred = predict_action(
                            features,
                            sample["state"],
                            seed,
                        )

                        torch.cuda.synchronize()

                        latency_ms = (
                            time.perf_counter() - tic
                        ) * 1000.0

                        pred = pred.float()
                        gt = sample[
                            "raw_actions"
                        ].float()

                        diff = pred - gt

                        dp = (
                            pred[:, 1:]
                            - pred[:, :-1]
                        )

                        dg = (
                            gt[:, 1:]
                            - gt[:, :-1]
                        )

                        ddp = (
                            dp[:, 1:]
                            - dp[:, :-1]
                        )

                        ddg = (
                            dg[:, 1:]
                            - dg[:, :-1]
                        )

                        pred_s = float(
                            dp.abs().mean().item()
                        )

                        demo_s = float(
                            dg.abs().mean().item()
                        )

                        metrics = {
                            "MAE": float(
                                diff.abs().mean().item()
                            ),
                            "MSE": float(
                                diff.square().mean().item()
                            ),
                            "Normalized_MAE": float(
                                (
                                    diff
                                    / AS.reshape(
                                        1, 1, ACTION_DIM
                                    )
                                ).abs().mean().item()
                            ),
                            "Delta_Error": float(
                                (dp - dg).abs().mean().item()
                            ),
                            "Second_Diff_Error": float(
                                (ddp - ddg).abs().mean().item()
                            ),
                            "Pred_Smoothness": pred_s,
                            "Demo_Smoothness": demo_s,
                            "Smoothness_Ratio":
                                pred_s / max(demo_s, 1e-8),
                        }

                        test_rows.append({
                            "Model": MODEL_NAME,
                            "Episode":
                                item["episode_index"],
                            "Frame":
                                item["frame_index"],
                            "Noise_Draw": draw,
                            "Latency_ms": latency_ms,
                            **metrics,
                        })

                if (window_id + 1) % 20 == 0:
                    log(
                        f"[TEST] {window_id+1}"
                        f"/{len(TEST_PLAN)}"
                    )

            TEST_DF = pd.DataFrame(
                test_rows
            )

            assert len(TEST_DF) == (
                177 * TEST_NOISE_DRAWS
            )

            assert TEST_DF[
                "Episode"
            ].nunique() == 59

            assert np.isfinite(
                TEST_DF.select_dtypes(
                    include=[np.number]
                ).to_numpy()
            ).all()

            mae = float(
                TEST_DF["MAE"].mean()
            )

            mse = float(
                TEST_DF["MSE"].mean()
            )

            TEST_SUMMARY = {
                "Model": MODEL_NAME,
                "Best_Step": BEST_STEP,
                "Test_Episodes": len(TEST_EPS),
                "Test_Windows": len(TEST_PLAN),
                "Predictions": len(TEST_DF),
                "MAE": mae,
                "MSE": mse,
                "RMSE": math.sqrt(mse),
                "Normalized_MAE": float(
                    TEST_DF["Normalized_MAE"].mean()
                ),
                "Test_Flow_Loss_t05": float(
                    np.mean(test_flow_losses)
                ),
                "Dataset_Revision": DATA_REVISION,
                "Split_SHA256": SPLIT_HASH,
            }

            ACTION_SUMMARY = {
                "Model": MODEL_NAME,
                "Delta_Error": float(
                    TEST_DF["Delta_Error"].mean()
                ),
                "Second_Diff_Error": float(
                    TEST_DF["Second_Diff_Error"].mean()
                ),
                "Pred_Smoothness": float(
                    TEST_DF["Pred_Smoothness"].mean()
                ),
                "Demo_Smoothness": float(
                    TEST_DF["Demo_Smoothness"].mean()
                ),
                "Smoothness_Ratio": float(
                    TEST_DF["Pred_Smoothness"].mean()
                    / max(
                        TEST_DF["Demo_Smoothness"].mean(),
                        1e-8
                    )
                ),
            }

            latency = TEST_DF[
                "Latency_ms"
            ].to_numpy()

            INFERENCE_SUMMARY = {
                "Model": MODEL_NAME,
                "Latency_Mean_ms": float(
                    np.mean(latency)
                ),
                "Latency_P50_ms": float(
                    np.percentile(latency, 50)
                ),
                "Latency_P95_ms": float(
                    np.percentile(latency, 95)
                ),
                "Throughput_chunks_per_s": float(
                    1000.0 / np.mean(latency)
                ),
                "Inference_Flow_Steps": FLOW_STEPS,
                "Noise_Draws_Per_Window":
                    TEST_NOISE_DRAWS,
                "GPU":
                    torch.cuda.get_device_name(DEVICE),
            }

            for filename, data in (
                (
                    "07_test_performance.csv",
                    TEST_SUMMARY,
                ),
                (
                    "08_action_stability.csv",
                    ACTION_SUMMARY,
                ),
                (
                    "09_inference_efficiency.csv",
                    INFERENCE_SUMMARY,
                ),
            ):
                pd.DataFrame(
                    [data]
                ).to_csv(
                    CSV_DIR / filename,
                    index=False,
                )

                log("[EXPORTED]", filename)

            log(
                "L8 HELD-OUT TEST: PASS",
                "531 predictions"
            )

    dist.barrier()

finally:
'''

source = source.replace(
    marker,
    patch,
    1,
)

script_path.write_text(
    source,
    encoding="utf-8",
)

compile(
    source,
    str(script_path),
    "exec",
)

print("CELL 12.5: PASS")
print("Final held-out test installed")
print("FAST-compatible CSV 07/08/09 enabled")
print("Script:", script_path)


CELL 12.5: PASS
Final held-out test installed
FAST-compatible CSV 07/08/09 enabled
Script: /kaggle/working/lfm2_l8_train_ddp.py


In [14]:

from pathlib import Path
import shutil

OUT = Path("/kaggle/working/L8_NEW_296_70_10_20")

LAST = OUT / "checkpoints" / "L8_last.pt"

if not LAST.exists():
    candidates = [
        p for p in Path("/kaggle/input").rglob("L8_last.pt")
        if p.parent.name == "checkpoints"
    ]

    if len(candidates) == 1:
        source = candidates[0].parent.parent

        for sub in ("checkpoints", "csv", "audit"):
            src = source / sub
            dst = OUT / sub

            if src.exists():
                shutil.copytree(
                    src,
                    dst,
                    dirs_exist_ok=True,
                )

        print("RESTORED:", candidates[0])

    elif len(candidates) > 1:
        raise RuntimeError(
            f"Multiple L8 checkpoints: {candidates}"
        )

    else:
        print("NEW LFM2 L8 TRAINING — step 0")

else:
    print("RESUME EXISTING:", LAST)


NEW LFM2 L8 TRAINING — step 0


In [15]:

import subprocess
import os

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    "/kaggle/working/lfm2_l8_train_ddp.py",
]

print("Launching LFM2 L8 DDP training")
print("GPUs: 2 x Tesla T4")
print("Max session: 11h30")
print("Automatic early stop: 15 minutes")
print("Checkpoint every 25 updates or 10 minutes")
print("Command:", " ".join(cmd), flush=True)

subprocess.run(
    cmd,
    check=True,
    env=os.environ.copy(),
)


Launching LFM2 L8 DDP training
GPUs: 2 x Tesla T4
Max session: 11h30
Automatic early stop: 15 minutes
Checkpoint every 25 updates or 10 minutes
Command: torchrun --standalone --nproc_per_node=2 /kaggle/working/lfm2_l8_train_ddp.py


W1103 12:21:43.010000 83 torch/distributed/run.py:851] 
W1103 12:21:43.010000 83 torch/distributed/run.py:851] *****************************************
W1103 12:21:43.010000 83 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1103 12:21:43.010000 83 torch/distributed/run.py:851] *****************************************


L8 CONFIGURATION: PASS
L8 CONFIGURATION: PASS
LFM2 L8 | DDP ranks: 2
Backbone: LiquidAI/LFM2-VL-450M
Layers: 8
AMP: torch.bfloat16


Fetching 78 files: 100%|██████████| 78/78 [01:05<00:00,  1.19it/s]


Dataset: 434945 frames
Split: 207 30 59
Split SHA256: c262a1e45efcb7f6c6c4a274a9671868440adc65e6eee1a3b737c0138811026e
Normalization frames: 302033


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1103 12:23:30.649390897 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()


... (18 outputs left out)

[SAVED] L8_last.pt step 3500 best MAE 5.9876391265127396
[SAVED] step=3500 reason=validation_completed
[L8 3520/4000] loss=0.17874 step_s=4.94 elapsed_min=307.5
[SAVED] L8_last.pt step 3525 best MAE 5.9876391265127396
[SAVED] step=3525 reason=periodic_or_time
[L8 3540/4000] loss=0.13330 step_s=5.19 elapsed_min=309.2
[SAVED] L8_last.pt step 3550 best MAE 5.9876391265127396
[SAVED] step=3550 reason=periodic_or_time
[L8 3560/4000] loss=0.13487 step_s=5.05 elapsed_min=310.9
[SAVED] L8_last.pt step 3575 best MAE 5.9876391265127396
[SAVED] step=3575 reason=periodic_or_time
[L8 3580/4000] loss=0.17926 step_s=4.97 elapsed_min=312.6
[L8 3600/4000] loss=0.14963 step_s=4.94 elapsed_min=314.3
[SAVED] L8_last.pt step 3600 best MAE 5.9876391265127396
[SAVED] step=3600 reason=periodic_or_time
[L8 3620/4000] loss=0.15896 step_s=4.99 elapsed_min=316.0
[SAVED] L8_last.pt step 3625 best MAE 5.9876391265127396
[SAVED] step=3625 reason=periodic_or_time
[L8 3640/4000] loss=0.13310 step_s=5.05 elapsed_min=31

/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


[FINAL TEST] Best step: 3500 Test windows: 177
[TEST] 20/177
[TEST] 40/177
[TEST] 60/177
[TEST] 80/177
[TEST] 100/177
[TEST] 120/177
[TEST] 140/177
[TEST] 160/177
[EXPORTED] 07_test_performance.csv
[EXPORTED] 08_action_stability.csv
[EXPORTED] 09_inference_efficiency.csv
L8 HELD-OUT TEST: PASS 531 predictions


CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '/kaggle/working/lfm2_l8_train_ddp.py'], returncode=0)

In [16]:

from pathlib import Path
import torch

out = Path("/kaggle/working/L8_NEW_296_70_10_20")

for path in sorted(
    (out / "checkpoints").glob("*.pt")
):
    saved = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
        mmap=True,
    )

    print("=" * 55)
    print("File:", path.name)
    print("Model:", saved.get("model"))
    print("Step:", saved.get("step"))
    print("Best validation MAE:", saved.get("best_mae"))
    print(
        "Size (GiB):",
        round(path.stat().st_size / 1024**3, 2),
    )

    assert saved["model"] == "L8"

    del saved

print("\nL8 CHECKPOINT AUDIT COMPLETE")
print("Output:", out)



L8 CHECKPOINT AUDIT COMPLETE
Output: /kaggle/working/L8_NEW_296_70_10_20
